# Chapter Continuation

### Basic Imports

In [1]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [3]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
            n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
                f"train loss: {history['train_losses'][-1]:.4f}, "
                f"train metric: {history['train_metrics'][-1]:.2%}, "
                f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

In [5]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Encoder-Only Transformers for Natural Language Understanding

> Pretraining BERT with MLM on the WikiText dataset, using the Transformers library:

In [6]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast

bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
config = BertConfig(  # adapt to training budget, and dataset size & complexity
    vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128)
bert = BertForMaskedLM(config)

In [7]:
from datasets import load_dataset

def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128,
                        padding="max_length")

mlm_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
mlm_dataset = mlm_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

In [8]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

args = TrainingArguments(output_dir="./my_bert", num_train_epochs=5,
                            per_device_train_batch_size=16,
                            report_to="none")  # to avoid requiring W&B or TB
mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm=True,
                                                mlm_probability=0.15)
trainer = Trainer(model=bert, args=args, train_dataset=mlm_dataset,
                    data_collator=mlm_collator)
trainer_output = trainer.train()

Step,Training Loss
500,8.885298
1000,7.502646
1500,7.312740
2000,7.221086
2500,7.169916
3000,7.130358
3500,7.124576
4000,7.065343
4500,7.075654
5000,7.014929


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
from transformers import pipeline

torch.manual_seed(42)
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
top_predictions = fill_mask("The capital of [MASK] is Rome.")
top_predictions[0]

{'score': 0.0394364558160305,
 'token': 1010,
 'token_str': ',',
 'sequence': 'the capital of, is rome.'}

In [10]:
del_vars(["bert_tokenizer", "config", "bert", "mlm_dataset", "args",
            "mlm_collator", "trainer", "trainer_output", "fill_mask",
            "top_predictions"])

In [11]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(["I like soccer", "Hello, World!"],
                    padding=True, return_tensors="pt")
outputs = model(**inputs)
cls_embedding = outputs.last_hidden_state[:, 0, :]
cls_embedding.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 768])

In [12]:
del_vars(["tokenizer", "model", "inputs", "outputs", "cls_embedding"])

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = ["She's shopping", "She bought some shoes", "She's working"]
embeddings = model.encode(sentences, convert_to_tensor=True)
similarities = model.similarity(embeddings, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
similarities

tensor([[1.0000, 0.6328, 0.5841],
        [0.6328, 1.0000, 0.3831],
        [0.5841, 0.3831, 1.0000]], device='cuda:0')

In [15]:
del_vars(["model", "sentences", "embeddings", "similarities"])

## Decoder-Only Transformers

In [36]:

from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.eos_token_id,
                             **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [38]:
prompt = "Scientists found a talking unicorn today. Here's the full story:"
generate(gpt2, gpt2_tokenizer, prompt)

"Scientists found a talking unicorn today. Here's the full story:\n\nThe unicorn was found in a field in the northern part of the state of New Mexico.\n\nThe unicorn was found in a field in the northern part of the state of New Mexico.\n\nThe unicorn was found in a field in"

In [39]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True)

"Scientists found a talking unicorn today. Here's the full story:\n\nThere aren't lots of other unicorns and they have been making their way across the United States since at least the 1800s, but this year there weren't a solitary unicorn on the land. Today, there are around 1,000."

In [40]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True, top_p=0.6)

"Scientists found a talking unicorn today. Here's the full story:\n\nThe Talking Unicorn\n\nIn this picture taken on Dec. 13, 2016, at the Zoo of America, the Talking Unicorn was the first living living creature in a living world.\n\nThe unicorn was a living creature of the zoo."

In [41]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [42]:
get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")

'London'

In [43]:
get_capital_city(gpt2, gpt2_tokenizer, "Mexico")

'Mexico City'

In [44]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
    for country in ("The UK", "Great Britain", "Big Britane")]

['London', 'London', 'London']

>However, it's far from perfect. In particular, it picked up many misconceptions from its training data. The capital cities of the following countries are respectively Canberra, Ottawa, Wellington, and Washington D.C., but GPT-2 gets it wrong:

In [45]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
    for country in ("Australia", "Canada", "New Zealand", "USA")]

['Sydney', 'Toronto', 'Auckland', 'New York']

And when the model is not sure, it often returns the name of the country (which is only the correct answer for about 10 countries)

In [46]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
    for country in ("Buthan", "Colombia", "Togo")]

['Buthan', 'Colombia', 'Togo']

In [47]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
    for country in ("hey", "yo", "j")]

['Paris', 'Paris', 'Paris']

In [48]:
del_vars(["model_id", "gpt2"])

In [49]:
try:
	access_token = open("hf-read-mistral.secret").read().strip()
except FileNotFoundError:
	from getpass import getpass
	access_token = getpass("Please enter your Hugging Face access token: ").strip()

In [50]:
from huggingface_hub import login

login(token=access_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [51]:
model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [52]:
torch.manual_seed(42)
generate(mistral7b, mistral7b_tokenizer, prompt, do_sample=True, top_p=0.6)

"Scientists found a talking unicorn today. Here's the full story:\n\n> The researchers say they found the unicorn by using a high-powered telescope and a special camera that detects light emitted by gases in the atmosphere of the distant star.\n>\n> The star is located about"

In [53]:
[get_capital_city(mistral7b, mistral7b_tokenizer, country)
    for country in ("Australia", "Canada", "New Zealand", "USA")]

['Canberra', 'Ottawa', 'Wellington', 'Washington DC']

In [54]:
[get_capital_city(mistral7b, mistral7b_tokenizer, country)
    for country in ("Buthan", "Colombia", "Togo")]

['Thimphu', 'Bogota', 'Lome']

In [55]:
prompt = "List some places I should visit in Paris."
generate(mistral7b, mistral7b_tokenizer, prompt)

'List some places I should visit in Paris.\n\nI’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.\n\nI’'

In [56]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""

In [57]:
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt,
                            max_new_tokens=100)
answer = extended_text[len(full_prompt):].strip()
print(answer)

The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.
Me: What's the best way to get around Paris?
Bob: The metro is the most efficient way to get around Paris.
Me: What's the best time of year to visit Paris?
Bob: The best time to visit Paris is in the spring or fall, when the weather is mild and the crowds are smaller


In [58]:
answer.split("\nMe: ")[0]

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [59]:
class BobTheChatbot:  # or ChatBob if you prefer
    def __init__(self, model, tokenizer, introduction=bob_introduction,
                    max_answer_length=10_000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length

    def chat(self, prompt):
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context,
                                max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context or
                len(answer) >= self.max_answer_length): break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [60]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bob.chat("List some places I should visit in Paris.")

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [61]:
bob.chat("Tell me more about the first place.")

'The Eiffel Tower is an iconic landmark in Paris and one of the most recognizable structures in the world. It was built in 1889 and stands at a height of 324 meters. Visitors can take an elevator to the top of the tower for panoramic views of the city.'

In [62]:
bob.chat("And Rome?")

"Rome is a city steeped in history and culture. It's home to many famous landmarks, including the Colosseum, the Pantheon, and the Trevi Fountain."

In [63]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("Tell me 5 jokes"))

KeyboardInterrupt: 

In [ ]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("How can I make cookies?"))

In [ ]:
bad_bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bad_bob.chat("I'd like to rob a bank. How should I prepare?")

> We need to fine-tune the model to make it more conversational and helpful. We can do this by fine-tuning it on a dataset containing the type of dialogue we'd like to see. This is typically done in two stages: SFT + DPO. The first is just regular supervised fine-tuning except we only compute the loss based on the model's answer. We will see an example using the TRL library shortly. DPO, is a bit more difficult (and interesting!), so let's look into it.

## DPO

In [ ]:
prompt = "The capital of Argentina is "
full_input = [prompt + "Buenos Aires", prompt + "Madrid"]
mistral7b_tokenizer.pad_token = mistral7b_tokenizer.eos_token
encodings = mistral7b_tokenizer(full_input, return_tensors="pt", padding=True)
encodings = encodings.to(device)
with torch.no_grad():
    logits = mistral7b(**encodings).logits  # shape [2, 8, 32768]